<a href="https://colab.research.google.com/github/CodeWithradhika/CSET343./blob/main/Lab7_DL_GridSearch_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSET343 - Lab Assignment 7
## Deep Learning Grid Search (Keras + Scikit-learn) on Pima Indians Diabetes

Note: scikeras is not used because it breaks with the new scikit-learn on Colab (`__sklearn_tags__` error). A small custom sklearn-compatible Keras wrapper is used instead.

## Imports

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
keras.utils.set_random_seed(SEED)

## Task 1: Data Acquisition and Loading

In [2]:
COLUMNS = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
           "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"]

URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
df = pd.read_csv(URL, header=None, names=COLUMNS)

print("Shape:", df.shape)
display(df.head())
print("\nClass distribution:\n", df["Outcome"].value_counts())

Shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1



Class distribution:
 Outcome
0    500
1    268
Name: count, dtype: int64


## Task 2: Data Preprocessing

In [3]:
# 0 is impossible in these columns -> treat as missing
zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
df[zero_as_missing] = df[zero_as_missing].replace(0, np.nan)
print("Missing values:\n", df.isnull().sum())

# Fill with median
df[zero_as_missing] = df[zero_as_missing].fillna(df[zero_as_missing].median())

X = df.drop("Outcome", axis=1).values
y = df["Outcome"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

# Scale (fit on train only -> no data leakage)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("\nTrain:", X_train.shape, "| Test:", X_test.shape)

Missing values:
 Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64

Train: (614, 8) | Test: (154, 8)


## Task 3: Deep Learning Grid Search
### Keras model + scikit-learn wrapper

In [4]:
def create_model(input_dim, hidden_units, activation, init_mode,
                 dropout_rate, learning_rate):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(hidden_units, activation=activation, kernel_initializer=init_mode),
        layers.Dropout(dropout_rate),
        layers.Dense(8, activation=activation, kernel_initializer=init_mode),
        layers.Dense(1, activation="sigmoid", kernel_initializer=init_mode),
    ])
    model.compile(loss="binary_crossentropy",
                  optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=["accuracy"])
    return model


class KerasClassifierWrapper(ClassifierMixin, BaseEstimator):
    """Minimal scikit-learn compatible wrapper around a Keras model."""
    def __init__(self, hidden_units=12, activation="relu",
                 init_mode="glorot_uniform", dropout_rate=0.0,
                 learning_rate=0.001, epochs=50, batch_size=16):
        self.hidden_units = hidden_units
        self.activation = activation
        self.init_mode = init_mode
        self.dropout_rate = dropout_rate
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size

    def fit(self, X, y):
        X = np.asarray(X, dtype="float32")
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        self.model_ = create_model(X.shape[1], self.hidden_units, self.activation,
                                   self.init_mode, self.dropout_rate,
                                   self.learning_rate)
        self.model_.fit(X, y, epochs=self.epochs,
                        batch_size=self.batch_size, verbose=0)
        return self

    def predict(self, X):
        probs = self.model_.predict(np.asarray(X, dtype="float32"), verbose=0)
        return (probs > 0.5).astype(int).ravel()

### Helper: run one grid search and carry the best params forward

In [5]:
best_params = dict(hidden_units=12, activation="relu", init_mode="glorot_uniform",
                   dropout_rate=0.0, learning_rate=0.001, epochs=50, batch_size=16)

def run_grid_search(title, param_grid):
    print("\n" + "=" * 60)
    print("Tuning:", title)
    print("=" * 60)

    clf = KerasClassifierWrapper(**best_params)
    grid = GridSearchCV(clf, param_grid, cv=3, scoring="accuracy", n_jobs=1)
    result = grid.fit(X_train, y_train)

    print(f"Best: {result.best_score_:.4f} using {result.best_params_}")
    for m, s, p in zip(result.cv_results_["mean_test_score"],
                       result.cv_results_["std_test_score"],
                       result.cv_results_["params"]):
        print(f"  {m:.4f} ({s:.4f}) with: {p}")

    best_params.update(result.best_params_)   # carry forward
    return result

### 1) Batch size and training epochs

In [6]:
r1 = run_grid_search("Batch Size & Epochs",
    {"batch_size": [10, 20, 40, 60, 80, 100],
     "epochs": [10, 50, 100]})


Tuning: Batch Size & Epochs


Best: 0.7850 using {'batch_size': 60, 'epochs': 50}
  0.7574 (0.0301) with: {'batch_size': 10, 'epochs': 10}
  0.7737 (0.0318) with: {'batch_size': 10, 'epochs': 50}
  0.7672 (0.0364) with: {'batch_size': 10, 'epochs': 100}
  0.7378 (0.0217) with: {'batch_size': 20, 'epochs': 10}
  0.7638 (0.0085) with: {'batch_size': 20, 'epochs': 50}
  0.7541 (0.0289) with: {'batch_size': 20, 'epochs': 100}
  0.7296 (0.0049) with: {'batch_size': 40, 'epochs': 10}
  0.7590 (0.0275) with: {'batch_size': 40, 'epochs': 50}
  0.7606 (0.0206) with: {'batch_size': 40, 'epochs': 100}
  0.7508 (0.0075) with: {'batch_size': 60, 'epochs': 10}
  0.7850 (0.0101) with: {'batch_size': 60, 'epochs': 50}
  0.7623 (0.0192) with: {'batch_size': 60, 'epochs': 100}
  0.6840 (0.0266) with: {'batch_size': 80, 'epochs': 10}
  0.7720 (0.0079) with: {'batch_size': 80, 'epochs': 50}
  0.7590 (0.0201) with: {'batch_size': 80, 'epochs': 100}
  0.6173 (0.0314) with: {'batch_size': 100, 'epochs': 10}
  0.7394 (0.0265) with: {'batc

### 2) Learning rate

In [7]:
r2 = run_grid_search("Learning Rate",
    {"learning_rate": [0.0001, 0.001, 0.01, 0.1]})


Tuning: Learning Rate
Best: 0.7671 using {'learning_rate': 0.001}
  0.6043 (0.0792) with: {'learning_rate': 0.0001}
  0.7671 (0.0148) with: {'learning_rate': 0.001}
  0.7362 (0.0237) with: {'learning_rate': 0.01}
  0.7297 (0.0109) with: {'learning_rate': 0.1}


### 3) Network weight initialization

In [8]:
r3 = run_grid_search("Weight Initialization",
    {"init_mode": ["random_uniform", "lecun_uniform", "random_normal", "zeros",
                   "glorot_normal", "glorot_uniform", "he_normal", "he_uniform"]})


Tuning: Weight Initialization
Best: 0.7850 using {'init_mode': 'random_normal'}
  0.7785 (0.0090) with: {'init_mode': 'random_uniform'}
  0.7623 (0.0197) with: {'init_mode': 'lecun_uniform'}
  0.7850 (0.0139) with: {'init_mode': 'random_normal'}
  0.6515 (0.0020) with: {'init_mode': 'zeros'}
  0.7704 (0.0208) with: {'init_mode': 'glorot_normal'}
  0.7785 (0.0041) with: {'init_mode': 'glorot_uniform'}
  0.7639 (0.0057) with: {'init_mode': 'he_normal'}
  0.7557 (0.0206) with: {'init_mode': 'he_uniform'}


### 4) Activation function

In [9]:
r4 = run_grid_search("Activation Function",
    {"activation": ["softmax", "softplus", "softsign", "relu",
                    "tanh", "sigmoid", "hard_sigmoid", "linear"]})


Tuning: Activation Function
Best: 0.7802 using {'activation': 'linear'}
  0.6515 (0.0020) with: {'activation': 'softmax'}
  0.7524 (0.0062) with: {'activation': 'softplus'}
  0.7801 (0.0040) with: {'activation': 'softsign'}
  0.7753 (0.0064) with: {'activation': 'relu'}
  0.7736 (0.0078) with: {'activation': 'tanh'}
  0.6515 (0.0020) with: {'activation': 'sigmoid'}
  0.6515 (0.0020) with: {'activation': 'hard_sigmoid'}
  0.7802 (0.0133) with: {'activation': 'linear'}


### 5) Dropout regularization

In [10]:
r5 = run_grid_search("Dropout Rate",
    {"dropout_rate": [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]})


Tuning: Dropout Rate
Best: 0.7785 using {'dropout_rate': 0.2}
  0.7769 (0.0161) with: {'dropout_rate': 0.0}
  0.7769 (0.0161) with: {'dropout_rate': 0.1}
  0.7785 (0.0146) with: {'dropout_rate': 0.2}
  0.7753 (0.0169) with: {'dropout_rate': 0.3}
  0.7753 (0.0169) with: {'dropout_rate': 0.4}
  0.7753 (0.0101) with: {'dropout_rate': 0.5}
  0.7720 (0.0087) with: {'dropout_rate': 0.6}


### 6) Number of neurons in the hidden layer

In [11]:
r6 = run_grid_search("Neurons in Hidden Layer",
    {"hidden_units": [1, 5, 10, 15, 20, 25, 30]})


Tuning: Neurons in Hidden Layer
Best: 0.7802 using {'hidden_units': 25}
  0.7573 (0.0043) with: {'hidden_units': 1}
  0.7769 (0.0118) with: {'hidden_units': 5}
  0.7753 (0.0101) with: {'hidden_units': 10}
  0.7769 (0.0161) with: {'hidden_units': 15}
  0.7769 (0.0161) with: {'hidden_units': 20}
  0.7802 (0.0169) with: {'hidden_units': 25}
  0.7769 (0.0156) with: {'hidden_units': 30}


## Final model with all best hyperparameters

In [12]:
print("FINAL BEST HYPERPARAMETERS")
for k, v in best_params.items():
    print(f"  {k}: {v}")

final_clf = KerasClassifierWrapper(**best_params)
final_clf.fit(X_train, y_train)

y_pred = final_clf.predict(X_test)
print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

FINAL BEST HYPERPARAMETERS
  hidden_units: 25
  activation: linear
  init_mode: random_normal
  dropout_rate: 0.2
  learning_rate: 0.001
  epochs: 50
  batch_size: 60

Test Accuracy: 0.7013

Classification Report:
               precision    recall  f1-score   support

           0       0.76      0.79      0.77       100
           1       0.58      0.54      0.56        54

    accuracy                           0.70       154
   macro avg       0.67      0.66      0.67       154
weighted avg       0.70      0.70      0.70       154

